In [1]:
import sps_databases
import subprocess
from cfbm.bm_data import get_data
import os
import scipy

In [2]:
# Before running any scripts that call schedule_workflow_job outside of a container, you'll need to run
!workflow workspace set champss.workspace.yml #puting !run the command in bash

Currently using champss workspace.
Locating workspace champss.workspace.yml
Workspace champss.workspace.yml not found.


In [2]:
#useful for seeing what input is needed for a fct
#help(CandidateViewerQuery.get_ratings)
#help(multidayfold_pipeline.main)

Currently using champss workspace.
Locating workspace champss.workspace.yml
Workspace champss.workspace.yml not found.


In [3]:
#Step_1-Query website and put them in a list
from sps_pipeline.candidate_viewer import CandidateViewerQuery
from multiday_search import multidayfold_pipeline
import os
from datetime import datetime, timedelta

# Database configuration
db_config = {
    'host': 'sps-archiver1',
    'user': 'automation',
    'port': 3306,
    'password': '',
    'database': 'champss'
}


#Creating a list of date
start_date = datetime(2026, 3, 5)
end_date   = datetime(2026, 3, 5)

folders = []
current = start_date

while current <= end_date:
    folders.append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)
#query = CandidateViewerQuery(survey="stackcands", db_config=db_config)
#candidates = query.get_metadata(folder="stack_0")
classifications = ['<faint>', 'NEW CANDIDATE']

all_candidates = []
with CandidateViewerQuery(survey='dailycands', db_config=db_config) as query:
    for folder in folders:
        print(f"\nProcessing folder: {folder}")

        for cls in classifications:
            try:
                candidates = query.get_ratings(
                    folder=folder,
                    classification=cls,
                    with_metadata=True
                )
            except Exception:
                print(f"No data for {folder}")
                continue  # skip this classification if query fails

            
            print(f"Found {len(candidates)} candidates for {cls} in {folder}")
            all_candidates.extend(candidates)

print("Query finished")


Processing folder: 2026-03-05
Found 4 candidates for <faint> in 2026-03-05
Found 0 candidates for NEW CANDIDATE in 2026-03-05
Query finished


In [ ]:
#Step_2-Run the multi-day fold
for cand in all_candidates:
    metadata = cand['metadata']
    input_file = metadata['input_file']
    outfile = f"/mnt/beegfs-client/processed/multiday/{metadata['file']}"
    print(input_file)

    # Skip if already folded
    if os.path.exists(outfile):
        print(f"Skipping {metadata['file']} — already folded")
        continue

    print(f"\nRunning multidayfold_pipeline for {metadata['file']}...")


    args = [
    "--candpath",
    input_file,
    "--db-name",
    "champss_processing",
    "--nday",
    "2", #We should put that to an optimal number instead and also add 
        #something to not run the multiday on the same candidate flagged as faint on mutliple day
    "--datpath",
    "/mnt/beegfs-client/raw/",
    "--foldpath",
    "/mnt/beegfs-client/processed/archives/",
    "--use-workflow"
]
    # Running the command
    try:
        fold_output = multidayfold_pipeline.main(
        args=args,
        standalone_mode=False
    )
        print(f"Fold finished. Output should be at: {foldpah}")#changed from outfile
    except Exception as e:
        print(f"Folding failed for: {metadata['file']}")
        print(e)

/mnt/beegfs-client/processed/mp_runs/daily_20260305/candidates/Multi_Pointing_Groups_f_13.658_DM_114.455_69aa4c5390c35a5cc64758d4.npz

Running multidayfold_pipeline for Multi_Pointing_Groups_f_13.658_DM_114.455_69aa4c5390c35a5cc64758d4...
Source md_125.72_61.89_13.657943_114.45 already in the follow-up source database.


/home/rtellier/miniconda3/envs/champss2/lib/python3.11/site-packages/pydantic/main.py:250: FutureWarning: WORKFLOW_TOKEN missing. Token auth will be required in the future.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


16 Mar 2026 18:17:30 UTC INFO      root Initial buckets entries: []

16 Mar 2026 18:17:30 UTC INFO      root Final buckets entries: []

/mnt/beegfs-client/raw/2026/02/11 24
/mnt/beegfs-client/raw/2026/02/24 23
Folding 2 days of data: ['20260211', '20260224']


...

16 Mar 2026 18:18:42 UTC INFO      root Removing finished service                                                  
                                  processing-fold-multiday-20260224-69b2d931ff2c065afe870e7b in state complete.

Finished multiday folding, beginning the coherent search


16 Mar 2026 18:18:42 UTC INFO      root Initial buckets entries: []

16 Mar 2026 18:18:42 UTC INFO      root Final buckets entries: []

16 Mar 2026 18:18:42 UTC INFO      root Creating Docker Service:                                                   
                                  {'image': 'sps-archiver1.chime:5000/champss_software:latest', 'name':            
                                  'processing-multiday-confirm-69b2d931ff2c065afe870e7b', 'command': 'workflow run 
                                  champss-multiday-confirm --tag 69b849823c2c678dcca8ba52 --site chime --lives 1   
                                  --sleep 1', 'env': ['CONTAINER_NAME={{.Task.Name}}',                             
                                  'NODE_NAME={{.Node.Hostname}}'], 'mode': {'replicated': {'Replicas': 1}},        
                                  'restart_policy': {'Condition': 'none', 'Delay': 0, 'MaxAttempts': 0, 'Window':  
                                  0}, 'labels': {'type': 'processing'}, 'constraints': ['node.labels.compute ==    
                                  true'], 'resources': {'Reservations': {'MemoryBytes': 64000000000}}, 'mounts':   
                                  [{'Target': '/var/run/docker.sock', 'Source': '/var/run/docker.sock', 'Type':    
                                  'bind', 'ReadOnly': False}, {'Target': '/dev/shm', 'Source': '', 'Type': 'tmpfs',
                                  'ReadOnly': False, 'TmpfsOptions': {'SizeBytes': 100000000000}}, {'Target':      
                                  '/mnt/beegfs-client/raw/', 'Source': '/mnt/beegfs-client/raw/', 'Type': 'bind',  
                                  'ReadOnly': False}, {'Target': '/mnt/beegfs-client/processed/archives/',         
                                  'Source': '/mnt/beegfs-client/processed/archives/', 'Type': 'bind', 'ReadOnly':  
                                  False}], 'networks': ['pipeline-network']}